In [101]:
import pandas as pd
import random
from multiprocessing import connection
from sqlalchemy import create_engine, text
import urllib


### Extract Data From Sql

In [102]:
# Parameters for connection
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=nasrulkhair\\SQLEXPRESS;"
    "DATABASE=TradingAnalyticsDB;"
    "Trusted_Connection=yes;"
)

# Create engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")



### Data Checking & Transformation

#### 1. Clients DF

In [103]:
# Read table from schema
dim_clients = pd.read_sql_table(
    table_name="dim_clients",
    con=engine,
    schema="raw_data"
)


print(dim_clients.head())
print(dim_clients.dtypes)

  client_id account_type country signup_date  is_active
0     C1000        Micro      TH  2024-11-23       True
1     C1001     Standard      MY  2024-02-27       True
2     C1002        Micro      ID  2024-01-13      False
3     C1003        Micro      MY  2024-05-20       True
4     C1004     Standard      PH  2024-05-05       True
client_id               object
account_type            object
country                 object
signup_date     datetime64[ns]
is_active                 bool
dtype: object


In [104]:
# mapping original country names to their codes

dim_clients['country'].value_counts()

country_mapping = {
    'MY': 'Malaysia',
    'SG': 'Singapore',
    'TH': 'Thailand',
    'ID': 'Indonesia',
    'PH': 'Philippines',
    'VN': 'Vietnam'
}
dim_clients['country_name'] = dim_clients['country'].map(country_mapping) 
dim_clients.head()  

# mapping True/False - active / not active
dim_clients['is_active'] =  dim_clients['is_active'].map({True:'Active', False: 'Inactive'})
 

# renam and rearrange columns
dim_clients = dim_clients.rename(columns={
    'client_id':'client_id',
    'account_type':'account_type',
    'country':'country_code',
    'country_name':'country_name',
    'signup_date':'signup_date',
    'is_active':'account_status'
})

dim_clients = dim_clients[['client_id', 'account_type', 'signup_date', 'account_status','country_code', 'country_name']]
dim_clients = dim_clients.copy()
dim_clients.head()


,client_id,account_type,signup_date,account_status,country_code,country_name
0,C1000,Micro,2024-11-23,Active,TH,Thailand
1,C1001,Standard,2024-02-27,Active,MY,Malaysia
2,C1002,Micro,2024-01-13,Inactive,ID,Indonesia
3,C1003,Micro,2024-05-20,Active,MY,Malaysia
4,C1004,Standard,2024-05-05,Active,PH,Philippines


#### Trades Df

In [105]:
query = "SELECT * FROM raw_data.fact_trades"
fact_trades = pd.read_sql(
    query,
    con=engine
)

print(fact_trades.head())

  trade_id client_id  symbol           open_time          close_time  volume  \
0    T1000     C1117  AUDUSD 2024-05-20 05:38:00 2024-05-20 09:15:00    4.97   
1    T1001     C1046  EURJPY 2024-02-24 18:24:00 2024-02-24 21:04:00    1.16   
2    T1002     C1116  USDCAD 2024-04-12 22:55:00 2024-04-13 03:20:00    2.96   
3    T1003     C1121  USDCHF 2024-04-25 04:07:00 2024-04-26 04:07:00    2.37   
4    T1004     C1027  AUDUSD 2024-05-30 18:54:00 2024-05-30 19:31:00    4.87   

   profit leverage  spread  
0    4.88   1:1000    1.11  
1  -38.65    1:500    1.55  
2   -5.55   1:1000    2.00  
3   48.82    1:200    2.69  
4  -95.64   1:1000    1.90  


In [106]:

# handling nulls - replacing close time with open time + avg tradiing duration
avg_duration = round(((fact_trades['close_time'] - fact_trades['open_time']).dt.total_seconds() / 60).mean(), 0)
print(avg_duration)

fact_trades['close_time']= fact_trades['close_time'].fillna(
    fact_trades['open_time'] + pd.to_timedelta(avg_duration, unit='m')
)

# new calculated columns
fact_trades['trade_duration_min'] = (fact_trades['close_time'] - fact_trades['open_time']).dt.total_seconds() / 60
fact_trades['trade_hour'] = fact_trades['open_time'].dt.hour

# mapping hour to trade session
session_labels = []
for hour in fact_trades['trade_hour']:
    if 6 <= hour < 14:
        session_labels.append('Sydney')
    elif 8 <= hour < 17:
        session_labels.append('Tokyo')
    elif hour >= 16 or hour < 1:
        session_labels.append('London')
    elif hour >= 21 or hour < 6:
        session_labels.append('New York')
    else:
        session_labels.append('Unknown')
        
fact_trades['trade_session'] = session_labels



992.0


In [107]:
# rearrange columns
fact_trades = fact_trades[['trade_id', 'client_id', 'symbol', 'open_time', 'close_time', 'trade_duration_min', 'trade_hour', 'leverage', 'spread',  'volume', 'profit',  'trade_session']]
fact_trades.head()

,trade_id,client_id,symbol,open_time,close_time,trade_duration_min,trade_hour,leverage,spread,volume,profit,trade_session
0,T1000,C1117,AUDUSD,2024-05-20 05:38:00,2024-05-20 09:15:00,217.0,5,1:1000,1.11,4.97,4.88,New York
1,T1001,C1046,EURJPY,2024-02-24 18:24:00,2024-02-24 21:04:00,160.0,18,1:500,1.55,1.16,-38.65,London
2,T1002,C1116,USDCAD,2024-04-12 22:55:00,2024-04-13 03:20:00,265.0,22,1:1000,2.00,2.96,-5.55,London
3,T1003,C1121,USDCHF,2024-04-25 04:07:00,2024-04-26 04:07:00,1440.0,4,1:200,2.69,2.37,48.82,New York
4,T1004,C1027,AUDUSD,2024-05-30 18:54:00,2024-05-30 19:31:00,37.0,18,1:1000,1.90,4.87,-95.64,London


#### Transaction Df

In [108]:
query = "SELECT * FROM raw_data.dim_transactions"
dim_transactions = pd.read_sql(
    query,
    con=engine
)

dim_transactions.head()

,transaction_id,client_id,tx_type,amount,tx_date,status
0,TX1000,C1094,Deposit,6140.19,2024-04-01,Rejected
1,TX1001,C1142,Withdrawal,6895.44,2024-02-26,Approved
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved
3,TX1003,C1017,Deposit,4240.85,2024-02-15,Rejected
4,TX1004,C1036,Withdrawal,1959.17,2024-03-06,Approved


In [109]:

# Rejected Reasons
deposit_rejected_reason = [
    'Mismatched account name',
    'Incomplete payment details',
    'Insufficient funds',
    'Technical errors or delays',
    'Unsupported payment method'
]

withdrawal_rejected_reason = [
    'Pending document approval',
    'Insufficient balance',
    'Suspicious or flagged activity'
]

dim_transactions['rejected_reason'] = dim_transactions.apply(
    lambda row: random.choice(deposit_rejected_reason) if row['tx_type'] == 'Deposit' and row['status'] == 'Rejected' else
                 random.choice(withdrawal_rejected_reason) if row['tx_type'] == 'Withdrawal' and row['status'] == 'Rejected' else
                 None, axis=1
)

dim_transactions.head()

,transaction_id,client_id,tx_type,amount,tx_date,status,rejected_reason
0,TX1000,C1094,Deposit,6140.19,2024-04-01,Rejected,Mismatched account name
1,TX1001,C1142,Withdrawal,6895.44,2024-02-26,Approved,None
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved,None
3,TX1003,C1017,Deposit,4240.85,2024-02-15,Rejected,Unsupported payment method
4,TX1004,C1036,Withdrawal,1959.17,2024-03-06,Approved,None


#### Accounts Snapshots Df

In [110]:
query = "SELECT * FROM raw_data.dim_snapshots"
dim_snapshots = pd.read_sql(
    query,
    con=engine
)

dim_snapshots.head()

,client_id,date,balance,equity,floating_pnl,swap,margin_used,free_margin
0,C1000,2024-06-01,-303.13,-303.13,0.00,0.00,0.0,-303.13
1,C1001,2024-06-01,58.15,58.15,0.00,0.00,0.0,58.15
2,C1002,2024-06-01,-30.86,-30.86,0.00,0.00,0.0,-30.86
3,C1003,2024-06-01,-10145.61,-10123.15,27.17,-4.71,9.1,-10132.25
4,C1004,2024-06-01,9766.78,9818.30,55.89,-4.37,50.2,9768.10


In [111]:
dim_snapshots.columns

Index(['client_id', 'date', 'balance', 'equity', 'floating_pnl', 'swap',
       'margin_used', 'free_margin'],
      dtype='object')

In [112]:
# Rearrange columns
dim_snapshots = dim_snapshots[['client_id', 'date', 'floating_pnl', 'swap', 'margin_used', 'free_margin','balance', 'equity']]

# Mapping account status based on - balance, free margin, and equity
dim_snapshots['account_status'] = dim_snapshots.apply(
    lambda x: 'Negative Balance' if x['balance'] < 0 else
              'Margin Call' if x['free_margin'] < 0 else
              'Active', axis = 1
)

dim_snapshots.head()

,client_id,date,floating_pnl,swap,margin_used,free_margin,balance,equity,account_status
0,C1000,2024-06-01,0.00,0.00,0.0,-303.13,-303.13,-303.13,Negative Balance
1,C1001,2024-06-01,0.00,0.00,0.0,58.15,58.15,58.15,Active
2,C1002,2024-06-01,0.00,0.00,0.0,-30.86,-30.86,-30.86,Negative Balance
3,C1003,2024-06-01,27.17,-4.71,9.1,-10132.25,-10145.61,-10123.15,Negative Balance
4,C1004,2024-06-01,55.89,-4.37,50.2,9768.10,9766.78,9818.30,Active


#### Cashflow Df

In [113]:
query = "SELECT * FROM raw_data.dim_cashflow"
dim_cashflow = pd.read_sql(
    query,
    con=engine
)

dim_cashflow.head()

,transaction_id,client_id,tx_type,amount,tx_date,status,description
0,TX1000,C1094,Deposit,6140.19,2024-04-01,Rejected,Deposit Made
1,TX1001,C1142,Withdrawal,6895.44,2024-02-26,Approved,Withdrawal requested
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved,Withdrawal requested
3,TX1003,C1017,Deposit,4240.85,2024-02-15,Rejected,Deposit Made
4,TX1004,C1036,Withdrawal,1959.17,2024-03-06,Approved,Withdrawal requested


#### Trade Analysis Df 

In [115]:
query = "SELECT * FROM raw_data.dim_trade_analysis"
dim_trade_analysis = pd.read_sql(
    query,
    con=engine
)
dim_trade_analysis.head()

,trade_id,client_id,symbol,entry,exit,pnl,duration_min,risk_reward,strategy
0,T1000,C1117,AUDUSD,2024-05-20 05:38:00,2024-05-20 09:15:00,4.88,217.0,0.49,Day Trading
1,T1001,C1046,EURJPY,2024-02-24 18:24:00,2024-02-24 21:04:00,-38.65,160.0,3.86,Day Trading
2,T1002,C1116,USDCAD,2024-04-12 22:55:00,2024-04-13 03:20:00,-5.55,265.0,0.55,Swing Trading
3,T1003,C1121,USDCHF,2024-04-25 04:07:00,2024-04-26 04:07:00,48.82,1440.0,4.88,Swing Trading
4,T1004,C1027,AUDUSD,2024-05-30 18:54:00,2024-05-30 19:31:00,-95.64,37.0,9.56,Day Trading


#### Client Risk Score Df

In [116]:
query = "SELECT * FROM raw_data.dim_client_risk_score"
dim_client_risk_score = pd.read_sql(
    query,
    con=engine
)
dim_client_risk_score.head()

,client_id,avg_drawdown,max_drawdown,avg_margin_used,win_rate_pct,trade_count,risk_score
0,C1000,0.000,0.00,0.00,27.272727,11,Medium
1,C1001,0.439,0.00,4.33,81.818182,11,Low
2,C1002,-92.515,-122.70,17.46,33.333333,12,Medium
3,C1003,27.170,27.17,9.10,66.666667,9,Low
4,C1004,55.890,55.89,50.20,66.666667,9,Low


#### Final Checking

In [117]:
print(dim_clients.info())
print(fact_trades.info())
print(dim_transactions.info())
print(dim_snapshots.info())
print(dim_cashflow.info())
print(dim_trade_analysis.info())
print(dim_client_risk_score.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   client_id       200 non-null    object        
 1   account_type    200 non-null    object        
 2   signup_date     200 non-null    datetime64[ns]
 3   account_status  200 non-null    object        
 4   country_code    200 non-null    object        
 5   country_name    200 non-null    object        
dtypes: datetime64[ns](1), object(5)
memory usage: 9.5+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2121 entries, 0 to 2120
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   trade_id            2121 non-null   object        
 1   client_id           2121 non-null   object        
 2   symbol              2121 non-null   object        
 3   open_time           2121

### Saving Data to Cleaned Data Schema -- SQL Server

In [119]:
with engine.connect() as conn:
    for schema in ['raw_data', 'cleaned_data', 'reportng']:
        conn.execute(text(f"""
                          IF NOT EXISTS (
                              SELECT * FROM sys.schemas WHERE name = '{schema}'
                          )
                          BEGIN
                              EXEC('CREATE SCHEMA {schema}')
                          END
                          """))
        
# Defining df to target schema
dataframes = {
    'dim_clients': (dim_clients, 'cleaned_data'),
    'fact_trades': (fact_trades, 'cleaned_data'),
    'dim_transactions': (dim_transactions, 'cleaned_data'),
    'dim_snapshots': (dim_snapshots, 'cleaned_data'),
    'dim_cashflow': (dim_cashflow, 'cleaned_data'),
    'dim_trade_analysis': (dim_trade_analysis, 'cleaned_data'),
    'dim_client_risk_score': (dim_client_risk_score, 'cleaned_data')
}

# Upload dataframes to SQL Server
for table_name, (df, schema) in dataframes.items():
    try:
        df.to_sql(
            table_name,
            con = engine,
            schema = schema,
            if_exists = 'replace',
            index = False)
        print(f'Uploaded {schema}.{table_name} successfully.')
    except Exception as e:
        print(f'Error yploading {schema}.{table_name}: {e}')

Uploaded cleaned_data.dim_clients successfully.
Uploaded cleaned_data.fact_trades successfully.
Uploaded cleaned_data.dim_transactions successfully.
Uploaded cleaned_data.dim_snapshots successfully.
Uploaded cleaned_data.dim_cashflow successfully.
Uploaded cleaned_data.dim_trade_analysis successfully.
Uploaded cleaned_data.dim_client_risk_score successfully.
